# Notebook 03 — Rules Engine Antifraud

## Fraud Graph Analytics  
### Motor de Regras Explicáveis para Prevenção a Fraudes Transacionais

Este notebook constrói a primeira versão do motor de regras antifraude do projeto **Fraud Graph Analytics**.

A proposta é transformar os achados exploratórios do Notebook 02 em regras interpretáveis, capazes de sinalizar transações suspeitas com base em:

- valor transacional;
- idade da conta;
- compartilhamento de dispositivo;
- concentração em beneficiários;
- risco de rede e dispositivo;
- rajadas transacionais;
- uso de canais digitais;
- combinação de múltiplos sinais.

O objetivo não é criar um sistema antifraude produtivo, mas sim um MVP analítico e explicável para portfólio, demonstrando raciocínio de negócio, engenharia de features e priorização de alertas.

## 1. Objetivo da Célula

### Objetivo

Configurar o ambiente inicial, carregar os dados sintéticos e preparar a base transacional enriquecida para aplicação das regras antifraude.

### Ações realizadas

- Importação das bibliotecas.
- Definição dos diretórios do projeto.
- Carregamento dos dados sintéticos.
- Enriquecimento das transações com atributos de conta, cliente, dispositivo, IP e beneficiário.
- Criação de variáveis auxiliares para aplicação das regras.

### Justificativa técnica

Um motor de regras antifraude depende de dados transacionais enriquecidos. As regras não devem olhar apenas para a transação isolada, mas também para contexto cadastral, relacionamento com dispositivos, concentração de beneficiários, risco de rede e comportamento temporal.

### Resultados esperados

Base transacional pronta para criação de features, aplicação das regras e geração de alertas explicáveis.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.float_format", "{:,.4f}".format)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
SYNTHETIC_DIR = DATA_DIR / "synthetic"
GOLD_DIR = DATA_DIR / "03-gold"
DOCS_DIR = PROJECT_ROOT / "docs"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

GOLD_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root:  {PROJECT_ROOT}")
print(f"Synthetic dir: {SYNTHETIC_DIR}")
print(f"Gold dir:      {GOLD_DIR}")
print(f"Docs dir:      {DOCS_DIR}")
print(f"Reports dir:   {REPORTS_DIR}")

Project root:  d:\_DS-Projects\Data-Science\fraud-graph-analytics
Synthetic dir: d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\synthetic
Gold dir:      d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold
Docs dir:      d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs
Reports dir:   d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports


## 2. Carregamento dos Dados Sintéticos

Nesta etapa serão carregados os dados gerados no Notebook 01.

As tabelas utilizadas são:

- clientes;
- contas;
- dispositivos;
- IPs;
- beneficiários;
- cartões;
- transações;
- labels de fraude.

Essas entidades serão usadas para criar features e aplicar regras antifraude explicáveis.

In [2]:
dataset_files = {
    "clientes": "clientes.parquet",
    "contas": "contas.parquet",
    "dispositivos": "dispositivos.parquet",
    "ips": "ips.parquet",
    "beneficiarios": "beneficiarios.parquet",
    "cartoes": "cartoes.parquet",
    "transacoes": "transacoes.parquet",
    "labels_fraude": "labels_fraude.parquet",
}

datasets = {}

for name, filename in dataset_files.items():
    path = SYNTHETIC_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

    datasets[name] = pd.read_parquet(path)

clientes = datasets["clientes"]
contas = datasets["contas"]
dispositivos = datasets["dispositivos"]
ips = datasets["ips"]
beneficiarios = datasets["beneficiarios"]
cartoes = datasets["cartoes"]
transacoes = datasets["transacoes"]
labels_fraude = datasets["labels_fraude"]

print("Datasets carregados com sucesso.")

Datasets carregados com sucesso.


In [3]:
summary_datasets = pd.DataFrame(
    [
        {
            "dataset": name,
            "linhas": df.shape[0],
            "colunas": df.shape[1],
            "memoria_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        }
        for name, df in datasets.items()
    ]
).sort_values("linhas", ascending=False)

summary_datasets

,dataset,linhas,colunas,memoria_mb
6,transacoes,80000,13,43.2600
7,labels_fraude,80000,4,18.9000
1,contas,6000,6,1.5900
0,clientes,5000,6,1.0600
2,dispositivos,4500,4,0.9600
5,cartoes,4000,5,1.0200
4,beneficiarios,3500,4,0.7700
3,ips,3000,4,0.6300


## 3. Enriquecimento da Base Transacional

Nesta etapa, a tabela de transações será enriquecida com informações das principais entidades relacionadas.

Esse enriquecimento permite que as regras considerem:

- perfil da conta;
- idade da conta;
- limite transacional;
- perfil do cliente;
- risco do dispositivo;
- risco de rede;
- características do beneficiário.

In [4]:
transacoes["data_hora"] = pd.to_datetime(transacoes["data_hora"])
contas["data_abertura"] = pd.to_datetime(contas["data_abertura"])
clientes["data_cadastro"] = pd.to_datetime(clientes["data_cadastro"])
cartoes["data_emissao"] = pd.to_datetime(cartoes["data_emissao"])

transacoes_enriched = (
    transacoes
    .merge(
        contas[
            [
                "conta_id",
                "cliente_id",
                "tipo_conta",
                "data_abertura",
                "status_conta",
                "limite_transacional_diario",
            ]
        ],
        left_on="conta_origem_id",
        right_on="conta_id",
        how="left",
    )
    .merge(
        clientes[
            [
                "cliente_id",
                "idade",
                "uf",
                "segmento",
                "data_cadastro",
                "score_cadastral",
            ]
        ],
        on="cliente_id",
        how="left",
    )
    .merge(dispositivos, on="device_id", how="left")
    .merge(ips, on="ip_id", how="left")
    .merge(beneficiarios, on="beneficiario_id", how="left")
)

transacoes_enriched["idade_conta_dias"] = (
    transacoes_enriched["data_hora"] - transacoes_enriched["data_abertura"]
).dt.days.clip(lower=0)

transacoes_enriched["hora"] = transacoes_enriched["data_hora"].dt.hour
transacoes_enriched["data"] = transacoes_enriched["data_hora"].dt.date
transacoes_enriched["ano_mes"] = transacoes_enriched["data_hora"].dt.to_period("M").astype(str)
transacoes_enriched["data_hora_hora"] = transacoes_enriched["data_hora"].dt.floor("h")

transacoes_enriched["valor_sobre_limite_diario"] = (
    transacoes_enriched["valor"] / transacoes_enriched["limite_transacional_diario"]
).replace([np.inf, -np.inf], np.nan)

transacoes_enriched.head()

,transacao_id,conta_origem_id,beneficiario_id,valor,data_hora,tipo_transacao,canal,device_id,ip_id,status_transacao,is_fraud,fraud_scenario,cartao_id,conta_id,cliente_id,tipo_conta,data_abertura,status_conta,limite_transacional_diario,idade,uf,segmento,data_cadastro,score_cadastral,tipo_device,sistema_operacional,fingerprint_risco,uf_origem,tipo_rede,risco_rede,tipo_beneficiario,banco_destino,uf_destino,idade_conta_dias,hora,data,ano_mes,data_hora_hora,valor_sobre_limite_diario
0,TX_00007707,CTA_001877,BEN_003322,203.3000,2025-01-01 03:05:51,pix,app,DEV_001817,IP_000089,aprovada,0,normal,None,CTA_001877,CLI_002620,corrente,2021-08-09,ativa,5000,74,MG,varejo,2025-09-11,803,mobile,iOS,baixo,CE,movel,alto,pessoa_fisica,banco_a,GO,1241,3,2025-01-01,2025-01,2025-01-01 03:00:00,0.0407
1,TX_00072064,CTA_003365,BEN_000265,275.9900,2025-01-01 03:13:25,pix,app,DEV_002968,IP_001202,aprovada,0,normal,None,CTA_003365,CLI_000516,corrente,2024-08-18,ativa,1000,24,SC,varejo,2021-10-05,541,mobile,Windows,baixo,ES,movel,baixo,pessoa_fisica,banco_b,PE,136,3,2025-01-01,2025-01,2025-01-01 03:00:00,0.2760
2,TX_00075764,CTA_000156,BEN_002832,11.5900,2025-01-01 04:00:23,pix,app,DEV_003600,IP_001519,aprovada,0,normal,None,CTA_000156,CLI_000344,corrente,2024-12-09,ativa,5000,48,SP,alta_renda,2025-06-19,657,mobile,Android,baixo,SP,residencial,baixo,conta_interna,banco_a,MG,23,4,2025-01-01,2025-01,2025-01-01 04:00:00,0.0023
3,TX_00023203,CTA_005491,BEN_001801,296.9700,2025-01-01 04:08:36,pix,app,DEV_001308,IP_001428,aprovada,0,normal,None,CTA_005491,CLI_002752,corrente,2022-08-16,ativa,20000,22,SP,aposentado,2021-08-21,603,mobile,Windows,baixo,MG,movel,medio,pessoa_juridica,banco_a,SP,869,4,2025-01-01,2025-01,2025-01-01 04:00:00,0.0148
4,TX_00001302,CTA_000997,BEN_002808,94.9800,2025-01-01 04:15:04,boleto,api,DEV_003878,IP_002967,aprovada,0,normal,None,CTA_000997,CLI_004486,corrente,2024-09-01,ativa,10000,40,PR,varejo,2024-07-28,593,mobile,iOS,baixo,GO,movel,baixo,pessoa_fisica,banco_c,PE,122,4,2025-01-01,2025-01,2025-01-01 04:00:00,0.0095


## 4. Engenharia de Features para Regras

Antes de aplicar as regras, serão criadas features agregadas que representam sinais comportamentais.

Essas features serão calculadas em diferentes níveis:

- dispositivo;
- beneficiário;
- IP;
- conta;
- janela temporal por conta;
- transação individual.

Essas variáveis serão usadas como insumos para o motor de regras.

In [5]:
device_features = (
    transacoes_enriched
    .groupby("device_id")
    .agg(
        device_qtd_transacoes=("transacao_id", "count"),
        device_contas_distintas=("conta_origem_id", "nunique"),
        device_clientes_distintos=("cliente_id", "nunique"),
        device_beneficiarios_distintos=("beneficiario_id", "nunique"),
        device_valor_total=("valor", "sum"),
        device_valor_medio=("valor", "mean"),
        device_taxa_fraude_sintetica=("is_fraud", "mean"),
    )
    .reset_index()
)

beneficiary_features = (
    transacoes_enriched
    .groupby("beneficiario_id")
    .agg(
        benef_qtd_transacoes=("transacao_id", "count"),
        benef_contas_origem_distintas=("conta_origem_id", "nunique"),
        benef_clientes_distintos=("cliente_id", "nunique"),
        benef_devices_distintos=("device_id", "nunique"),
        benef_valor_total=("valor", "sum"),
        benef_valor_medio=("valor", "mean"),
        benef_taxa_fraude_sintetica=("is_fraud", "mean"),
    )
    .reset_index()
)

ip_features = (
    transacoes_enriched
    .groupby("ip_id")
    .agg(
        ip_qtd_transacoes=("transacao_id", "count"),
        ip_contas_distintas=("conta_origem_id", "nunique"),
        ip_clientes_distintos=("cliente_id", "nunique"),
        ip_devices_distintos=("device_id", "nunique"),
        ip_valor_total=("valor", "sum"),
        ip_taxa_fraude_sintetica=("is_fraud", "mean"),
    )
    .reset_index()
)

account_hour_features = (
    transacoes_enriched
    .groupby(["conta_origem_id", "data_hora_hora"])
    .agg(
        qtd_transacoes_conta_hora=("transacao_id", "count"),
        valor_total_conta_hora=("valor", "sum"),
        qtd_beneficiarios_conta_hora=("beneficiario_id", "nunique"),
    )
    .reset_index()
)

account_day_features = (
    transacoes_enriched
    .groupby(["conta_origem_id", "data"])
    .agg(
        qtd_transacoes_conta_dia=("transacao_id", "count"),
        valor_total_conta_dia=("valor", "sum"),
        qtd_beneficiarios_conta_dia=("beneficiario_id", "nunique"),
        qtd_devices_conta_dia=("device_id", "nunique"),
    )
    .reset_index()
)

display(device_features.head())
display(beneficiary_features.head())
display(ip_features.head())
display(account_hour_features.head())
display(account_day_features.head())

,device_id,device_qtd_transacoes,device_contas_distintas,device_clientes_distintos,device_beneficiarios_distintos,device_valor_total,device_valor_medio,device_taxa_fraude_sintetica
0,DEV_000001,16,15,15,16,"4,719.3400",294.9588,0.0000
1,DEV_000002,16,16,16,16,"17,149.0800","1,071.8175",0.1875
2,DEV_000003,16,16,16,15,"5,417.5700",338.5981,0.0625
3,DEV_000004,14,14,14,14,"14,678.9800","1,048.4986",0.0714
4,DEV_000005,10,10,10,10,"3,678.6900",367.8690,0.0000


,beneficiario_id,benef_qtd_transacoes,benef_contas_origem_distintas,benef_clientes_distintos,benef_devices_distintos,benef_valor_total,benef_valor_medio,benef_taxa_fraude_sintetica
0,BEN_000001,28,28,28,28,"51,212.9500","1,829.0339",0.0714
1,BEN_000002,26,26,26,26,"15,587.8000",599.5308,0.0385
2,BEN_000003,21,21,20,21,"7,445.2500",354.5357,0.0952
3,BEN_000004,23,23,23,23,"53,862.7600","2,341.8591",0.0870
4,BEN_000005,17,17,17,17,"6,976.3800",410.3753,0.0588


,ip_id,ip_qtd_transacoes,ip_contas_distintas,ip_clientes_distintos,ip_devices_distintos,ip_valor_total,ip_taxa_fraude_sintetica
0,IP_000001,19,19,19,19,"12,078.8600",0.0000
1,IP_000002,27,27,27,27,"16,577.2800",0.0741
2,IP_000003,25,25,25,25,"11,086.1000",0.0400
3,IP_000004,26,26,26,26,"13,532.2700",0.0385
4,IP_000005,13,13,13,13,"7,153.2100",0.0769


,conta_origem_id,data_hora_hora,qtd_transacoes_conta_hora,valor_total_conta_hora,qtd_beneficiarios_conta_hora
0,CTA_000001,2025-01-18 14:00:00,1,516.5200,1
1,CTA_000001,2025-01-26 10:00:00,1,353.7400,1
2,CTA_000001,2025-02-18 22:00:00,1,285.9300,1
3,CTA_000001,2025-04-05 06:00:00,1,409.2800,1
4,CTA_000001,2025-05-02 07:00:00,1,631.7400,1


,conta_origem_id,data,qtd_transacoes_conta_dia,valor_total_conta_dia,qtd_beneficiarios_conta_dia,qtd_devices_conta_dia
0,CTA_000001,2025-01-18,1,516.5200,1,1
1,CTA_000001,2025-01-26,1,353.7400,1,1
2,CTA_000001,2025-02-18,1,285.9300,1,1
3,CTA_000001,2025-04-05,1,409.2800,1,1
4,CTA_000001,2025-05-02,1,631.7400,1,1


In [6]:
rules_base = (
    transacoes_enriched
    .merge(device_features, on="device_id", how="left")
    .merge(beneficiary_features, on="beneficiario_id", how="left")
    .merge(ip_features, on="ip_id", how="left")
    .merge(account_hour_features, on=["conta_origem_id", "data_hora_hora"], how="left")
    .merge(account_day_features, on=["conta_origem_id", "data"], how="left")
)

rules_base.head()

,transacao_id,conta_origem_id,beneficiario_id,valor,data_hora,tipo_transacao,canal,device_id,ip_id,status_transacao,is_fraud,fraud_scenario,cartao_id,conta_id,cliente_id,tipo_conta,data_abertura,status_conta,limite_transacional_diario,idade,uf,segmento,data_cadastro,score_cadastral,tipo_device,sistema_operacional,fingerprint_risco,uf_origem,tipo_rede,risco_rede,tipo_beneficiario,banco_destino,uf_destino,idade_conta_dias,hora,data,ano_mes,data_hora_hora,valor_sobre_limite_diario,device_qtd_transacoes,device_contas_distintas,device_clientes_distintos,device_beneficiarios_distintos,device_valor_total,device_valor_medio,device_taxa_fraude_sintetica,benef_qtd_transacoes,benef_contas_origem_distintas,benef_clientes_distintos,benef_devices_distintos,benef_valor_total,benef_valor_medio,benef_taxa_fraude_sintetica,ip_qtd_transacoes,ip_contas_distintas,ip_clientes_distintos,ip_devices_distintos,ip_valor_total,ip_taxa_fraude_sintetica,qtd_transacoes_conta_hora,valor_total_conta_hora,qtd_beneficiarios_conta_hora,qtd_transacoes_conta_dia,valor_total_conta_dia,qtd_beneficiarios_conta_dia,qtd_devices_conta_dia
0,TX_00007707,CTA_001877,BEN_003322,203.3000,2025-01-01 03:05:51,pix,app,DEV_001817,IP_000089,aprovada,0,normal,None,CTA_001877,CLI_002620,corrente,2021-08-09,ativa,5000,74,MG,varejo,2025-09-11,803,mobile,iOS,baixo,CE,movel,alto,pessoa_fisica,banco_a,GO,1241,3,2025-01-01,2025-01,2025-01-01 03:00:00,0.0407,21,21,21,21,"8,564.2800",407.8229,0.0000,24,24,24,24,"8,964.8900",373.5371,0.0417,31,31,31,31,"13,928.8700",0.0968,1,203.3000,1,1,203.3000,1,1
1,TX_00072064,CTA_003365,BEN_000265,275.9900,2025-01-01 03:13:25,pix,app,DEV_002968,IP_001202,aprovada,0,normal,None,CTA_003365,CLI_000516,corrente,2024-08-18,ativa,1000,24,SC,varejo,2021-10-05,541,mobile,Windows,baixo,ES,movel,baixo,pessoa_fisica,banco_b,PE,136,3,2025-01-01,2025-01,2025-01-01 03:00:00,0.2760,32,32,32,32,"27,052.0900",845.3778,0.0625,22,22,22,21,"8,625.2100",392.0550,0.0455,28,28,28,28,"19,608.6000",0.1071,1,275.9900,1,1,275.9900,1,1
2,TX_00075764,CTA_000156,BEN_002832,11.5900,2025-01-01 04:00:23,pix,app,DEV_003600,IP_001519,aprovada,0,normal,None,CTA_000156,CLI_000344,corrente,2024-12-09,ativa,5000,48,SP,alta_renda,2025-06-19,657,mobile,Android,baixo,SP,residencial,baixo,conta_interna,banco_a,MG,23,4,2025-01-01,2025-01,2025-01-01 04:00:00,0.0023,18,18,18,18,"7,943.9800",441.3322,0.0556,25,25,25,25,"11,225.6700",449.0268,0.0400,22,22,22,22,"8,337.6400",0.0455,1,11.5900,1,1,11.5900,1,1
3,TX_00023203,CTA_005491,BEN_001801,296.9700,2025-01-01 04:08:36,pix,app,DEV_001308,IP_001428,aprovada,0,normal,None,CTA_005491,CLI_002752,corrente,2022-08-16,ativa,20000,22,SP,aposentado,2021-08-21,603,mobile,Windows,baixo,MG,movel,medio,pessoa_juridica,banco_a,SP,869,4,2025-01-01,2025-01,2025-01-01 04:00:00,0.0148,23,23,23,22,"13,867.5300",602.9361,0.0435,24,24,24,24,"14,078.9500",586.6229,0.0417,21,21,21,21,"28,712.9100",0.1905,1,296.9700,1,1,296.9700,1,1
4,TX_00001302,CTA_000997,BEN_002808,94.9800,2025-01-01 04:15:04,boleto,api,DEV_003878,IP_002967,aprovada,0,normal,None,CTA_000997,CLI_004486,corrente,2024-09-01,ativa,10000,40,PR,varejo,2024-07-28,593,mobile,iOS,baixo,GO,movel,baixo,pessoa_fisica,banco_c,PE,122,4,2025-01-01,2025-01,2025-01-01 04:00:00,0.0095,16,16,16,16,"14,359.4900",897.4681,0.0625,29,29,29,29,"24,147.5200",832.6731,0.1034,28,28,27,28,"13,888.2600",0.0714,1,94.9800,1,1,94.9800,1,1


## 5. Catálogo de Regras Antifraude

O catálogo de regras define as regras candidatas do MVP.

Cada regra possui:

- identificador;
- nome;
- descrição;
- racional de negócio;
- severidade;
- pontuação;
- tipo de sinal.

Esse catálogo será usado para documentação, rastreabilidade e explicabilidade dos alertas.

In [7]:
rules_catalog = pd.DataFrame(
    [
        {
            "rule_id": "R001",
            "rule_name": "alto_valor_transacional",
            "descricao": "Transação com valor acima do percentil 95 da base.",
            "racional_negocio": "Transações muito acima do padrão geral podem indicar tentativa de movimentação atípica.",
            "severidade": "media",
            "pontos": 15,
            "tipo_sinal": "valor",
        },
        {
            "rule_id": "R002",
            "rule_name": "valor_acima_limite_diario",
            "descricao": "Transação cujo valor representa mais de 80% do limite transacional diário da conta.",
            "racional_negocio": "Uso intensivo do limite pode indicar maior risco operacional.",
            "severidade": "media",
            "pontos": 12,
            "tipo_sinal": "valor_limite",
        },
        {
            "rule_id": "R003",
            "rule_name": "conta_nova_alto_valor",
            "descricao": "Conta com até 60 dias realizando transação acima do percentil 90.",
            "racional_negocio": "Contas novas com transações elevadas são relevantes em investigações de fraude transacional.",
            "severidade": "alta",
            "pontos": 22,
            "tipo_sinal": "conta",
        },
        {
            "rule_id": "R004",
            "rule_name": "dispositivo_compartilhado",
            "descricao": "Dispositivo utilizado por cinco ou mais contas distintas.",
            "racional_negocio": "Um mesmo dispositivo operando muitas contas pode indicar coordenação, intermediação ou tomada de conta.",
            "severidade": "alta",
            "pontos": 20,
            "tipo_sinal": "device",
        },
        {
            "rule_id": "R005",
            "rule_name": "beneficiario_concentrador",
            "descricao": "Beneficiário recebeu transações de 25 ou mais contas distintas.",
            "racional_negocio": "Recebedores com muitas origens podem atuar como pontos de concentração de valores suspeitos.",
            "severidade": "alta",
            "pontos": 20,
            "tipo_sinal": "beneficiario",
        },
        {
            "rule_id": "R006",
            "rule_name": "rede_ou_device_alto_risco",
            "descricao": "Transação originada de IP de alto risco ou dispositivo com fingerprint de alto risco.",
            "racional_negocio": "Sinais técnicos de risco aumentam a prioridade investigativa.",
            "severidade": "media",
            "pontos": 14,
            "tipo_sinal": "risco_tecnico",
        },
        {
            "rule_id": "R007",
            "rule_name": "rajada_transacional_horaria",
            "descricao": "Conta realizou cinco ou mais transações na mesma hora.",
            "racional_negocio": "Múltiplas transações em curto intervalo podem indicar tentativa de escoamento rápido de valores.",
            "severidade": "alta",
            "pontos": 22,
            "tipo_sinal": "velocidade",
        },
        {
            "rule_id": "R008",
            "rule_name": "muitos_beneficiarios_no_dia",
            "descricao": "Conta enviou valores para cinco ou mais beneficiários distintos no mesmo dia.",
            "racional_negocio": "Pulverização de destinos em uma mesma janela diária pode ser sinal de comportamento atípico.",
            "severidade": "media",
            "pontos": 12,
            "tipo_sinal": "pulverizacao",
        },
        {
            "rule_id": "R009",
            "rule_name": "ip_compartilhado_multiplas_contas",
            "descricao": "IP utilizado por dez ou mais contas distintas.",
            "racional_negocio": "Um mesmo IP transacionando para muitas contas pode indicar estrutura coordenada ou origem técnica suspeita.",
            "severidade": "media",
            "pontos": 12,
            "tipo_sinal": "ip",
        },
        {
            "rule_id": "R010",
            "rule_name": "canal_digital_alto_valor",
            "descricao": "Transação de alto valor realizada por canal digital.",
            "racional_negocio": "Canais digitais com valores elevados podem exigir maior priorização operacional.",
            "severidade": "baixa",
            "pontos": 8,
            "tipo_sinal": "canal",
        },
    ]
)

rules_catalog

,rule_id,rule_name,descricao,racional_negocio,severidade,pontos,tipo_sinal
0,R001,alto_valor_transacional,Transação com valor acima do percentil 95 da base.,Transações muito acima do padrão geral podem indicar tentativa de movimentação atípica.,media,15,valor
1,R002,valor_acima_limite_diario,Transação cujo valor representa mais de 80% do limite transacional diário da conta.,Uso intensivo do limite pode indicar maior risco operacional.,media,12,valor_limite
2,R003,conta_nova_alto_valor,Conta com até 60 dias realizando transação acima do percentil 90.,Contas novas com transações elevadas são relevantes em investigações de fraude transacional.,alta,22,conta
3,R004,dispositivo_compartilhado,Dispositivo utilizado por cinco ou mais contas distintas.,"Um mesmo dispositivo operando muitas contas pode indicar coordenação, intermediação ou tomada de conta.",alta,20,device
4,R005,beneficiario_concentrador,Beneficiário recebeu transações de 25 ou mais contas distintas.,Recebedores com muitas origens podem atuar como pontos de concentração de valores suspeitos.,alta,20,beneficiario
5,R006,rede_ou_device_alto_risco,Transação originada de IP de alto risco ou dispositivo com fingerprint de alto risco.,Sinais técnicos de risco aumentam a prioridade investigativa.,media,14,risco_tecnico
6,R007,rajada_transacional_horaria,Conta realizou cinco ou mais transações na mesma hora.,Múltiplas transações em curto intervalo podem indicar tentativa de escoamento rápido de valores.,alta,22,velocidade
7,R008,muitos_beneficiarios_no_dia,Conta enviou valores para cinco ou mais beneficiários distintos no mesmo dia.,Pulverização de destinos em uma mesma janela diária pode ser sinal de comportamento atípico.,media,12,pulverizacao
8,R009,ip_compartilhado_multiplas_contas,IP utilizado por dez ou mais contas distintas.,Um mesmo IP transacionando para muitas contas pode indicar estrutura coordenada ou origem técnica suspeita.,media,12,ip
9,R010,canal_digital_alto_valor,Transação de alto valor realizada por canal digital.,Canais digitais com valores elevados podem exigir maior priorização operacional.,baixa,8,canal


## 6. Aplicação das Regras

Nesta etapa serão criadas flags booleanas para cada regra do catálogo.

As regras serão aplicadas diretamente sobre a base transacional enriquecida.

In [8]:
value_p90 = rules_base["valor"].quantile(0.90)
value_p95 = rules_base["valor"].quantile(0.95)

print(f"Percentil 90 de valor: R$ {value_p90:,.2f}")
print(f"Percentil 95 de valor: R$ {value_p95:,.2f}")

Percentil 90 de valor: R$ 1,473.18
Percentil 95 de valor: R$ 2,796.95


In [9]:
rules_scored = rules_base.copy()

rules_scored["R001_alto_valor_transacional"] = rules_scored["valor"] >= value_p95

rules_scored["R002_valor_acima_limite_diario"] = (
    rules_scored["valor_sobre_limite_diario"] >= 0.80
)

rules_scored["R003_conta_nova_alto_valor"] = (
    (rules_scored["idade_conta_dias"] <= 60)
    & (rules_scored["valor"] >= value_p90)
)

rules_scored["R004_dispositivo_compartilhado"] = (
    rules_scored["device_contas_distintas"] >= 5
)

rules_scored["R005_beneficiario_concentrador"] = (
    rules_scored["benef_contas_origem_distintas"] >= 25
)

rules_scored["R006_rede_ou_device_alto_risco"] = (
    (rules_scored["risco_rede"] == "alto")
    | (rules_scored["fingerprint_risco"] == "alto")
)

rules_scored["R007_rajada_transacional_horaria"] = (
    rules_scored["qtd_transacoes_conta_hora"] >= 5
)

rules_scored["R008_muitos_beneficiarios_no_dia"] = (
    rules_scored["qtd_beneficiarios_conta_dia"] >= 5
)

rules_scored["R009_ip_compartilhado_multiplas_contas"] = (
    rules_scored["ip_contas_distintas"] >= 10
)

rules_scored["R010_canal_digital_alto_valor"] = (
    (rules_scored["canal"].isin(["app", "internet_banking", "api"]))
    & (rules_scored["valor"] >= value_p90)
)

rule_flag_columns = [
    "R001_alto_valor_transacional",
    "R002_valor_acima_limite_diario",
    "R003_conta_nova_alto_valor",
    "R004_dispositivo_compartilhado",
    "R005_beneficiario_concentrador",
    "R006_rede_ou_device_alto_risco",
    "R007_rajada_transacional_horaria",
    "R008_muitos_beneficiarios_no_dia",
    "R009_ip_compartilhado_multiplas_contas",
    "R010_canal_digital_alto_valor",
]

rules_scored[rule_flag_columns].mean().sort_values(ascending=False).to_frame("taxa_acionamento")

,taxa_acionamento
R009_ip_compartilhado_multiplas_contas,1.0000
R004_dispositivo_compartilhado,1.0000
R005_beneficiario_concentrador,0.3934
R006_rede_ou_device_alto_risco,0.1190
R010_canal_digital_alto_valor,0.0890
R002_valor_acima_limite_diario,0.0606
R001_alto_valor_transacional,0.0500
R003_conta_nova_alto_valor,0.0243
R008_muitos_beneficiarios_no_dia,0.0150
R007_rajada_transacional_horaria,0.0148


## 7. Cálculo do Score de Regras

Cada regra acionada adicionará pontos ao score antifraude da transação.

O score será calculado como a soma ponderada dos pontos das regras acionadas, limitado a 100 pontos.

A partir desse score, cada transação será classificada em uma faixa de risco:

- baixo;
- médio;
- alto;
- crítico.

In [10]:
rule_points = {
    "R001_alto_valor_transacional": 15,
    "R002_valor_acima_limite_diario": 12,
    "R003_conta_nova_alto_valor": 22,
    "R004_dispositivo_compartilhado": 20,
    "R005_beneficiario_concentrador": 20,
    "R006_rede_ou_device_alto_risco": 14,
    "R007_rajada_transacional_horaria": 22,
    "R008_muitos_beneficiarios_no_dia": 12,
    "R009_ip_compartilhado_multiplas_contas": 12,
    "R010_canal_digital_alto_valor": 8,
}

rules_scored["qtd_regras_acionadas"] = rules_scored[rule_flag_columns].sum(axis=1)

rules_scored["rule_score_raw"] = 0

for rule_col, points in rule_points.items():
    rules_scored["rule_score_raw"] += rules_scored[rule_col].astype(int) * points

rules_scored["rule_score"] = rules_scored["rule_score_raw"].clip(upper=100)

rules_scored[["transacao_id", "qtd_regras_acionadas", "rule_score"]].head()

,transacao_id,qtd_regras_acionadas,rule_score
0,TX_00007707,3,46
1,TX_00072064,2,32
2,TX_00075764,3,52
3,TX_00023203,2,32
4,TX_00001302,3,52


In [11]:
def assign_risk_band(score: float) -> str:
    if score >= 70:
        return "critico"
    if score >= 45:
        return "alto"
    if score >= 20:
        return "medio"
    return "baixo"


rules_scored["risk_band"] = rules_scored["rule_score"].apply(assign_risk_band)

rules_scored["alerta_gerado"] = rules_scored["rule_score"] >= 20

rules_scored[["rule_score", "risk_band", "alerta_gerado"]].value_counts().reset_index(name="qtd")

,rule_score,risk_band,alerta_gerado,qtd
0,32,medio,True,38941
1,52,alto,True,24420
2,46,alto,True,4771
3,66,alto,True,3298
4,100,critico,True,1376
5,40,medio,True,998
6,89,critico,True,881
7,60,alto,True,780
8,44,medio,True,555
9,67,alto,True,429


## 8. Explicabilidade dos Alertas

Nesta etapa será criada uma explicação textual para cada transação.

A explicação descreve quais regras foram acionadas e por que a transação recebeu determinada pontuação.

Essa camada é importante porque prevenção a fraudes não exige apenas identificar risco, mas também justificar a priorização para investigação.

In [12]:
rule_explanations = {
    "R001_alto_valor_transacional": "valor da transação acima do percentil 95 da base",
    "R002_valor_acima_limite_diario": "valor representa parcela elevada do limite transacional diário",
    "R003_conta_nova_alto_valor": "conta nova realizando transação de alto valor",
    "R004_dispositivo_compartilhado": "dispositivo utilizado por múltiplas contas",
    "R005_beneficiario_concentrador": "beneficiário recebe valores de muitas contas distintas",
    "R006_rede_ou_device_alto_risco": "IP ou dispositivo possui sinal técnico de alto risco",
    "R007_rajada_transacional_horaria": "conta realizou muitas transações na mesma hora",
    "R008_muitos_beneficiarios_no_dia": "conta enviou valores para muitos beneficiários no mesmo dia",
    "R009_ip_compartilhado_multiplas_contas": "IP utilizado por múltiplas contas distintas",
    "R010_canal_digital_alto_valor": "transação de alto valor realizada por canal digital",
}


def build_alert_explanation(row: pd.Series) -> str:
    triggered = [
        explanation
        for rule, explanation in rule_explanations.items()
        if bool(row[rule])
    ]

    if not triggered:
        return "Nenhuma regra antifraude relevante foi acionada."

    joined = "; ".join(triggered)

    return (
        f"Score {row['rule_score']:.0f}/100 — risco {row['risk_band']}. "
        f"Regras acionadas: {joined}."
    )


rules_scored["alert_explanation"] = rules_scored.apply(build_alert_explanation, axis=1)

rules_scored[
    [
        "transacao_id",
        "valor",
        "fraud_scenario",
        "is_fraud",
        "qtd_regras_acionadas",
        "rule_score",
        "risk_band",
        "alert_explanation",
    ]
].head(10)

,transacao_id,valor,fraud_scenario,is_fraud,qtd_regras_acionadas,rule_score,risk_band,alert_explanation
0,TX_00007707,203.3000,normal,0,3,46,alto,Score 46/100 — risco alto. Regras acionadas: dispositivo utilizado por múltiplas contas; IP ou dispositivo possui sinal técnico de alto risco; IP utilizado por múltiplas contas...
1,TX_00072064,275.9900,normal,0,2,32,medio,Score 32/100 — risco medio. Regras acionadas: dispositivo utilizado por múltiplas contas; IP utilizado por múltiplas contas distintas.
2,TX_00075764,11.5900,normal,0,3,52,alto,Score 52/100 — risco alto. Regras acionadas: dispositivo utilizado por múltiplas contas; beneficiário recebe valores de muitas contas distintas; IP utilizado por múltiplas cont...
3,TX_00023203,296.9700,normal,0,2,32,medio,Score 32/100 — risco medio. Regras acionadas: dispositivo utilizado por múltiplas contas; IP utilizado por múltiplas contas distintas.
4,TX_00001302,94.9800,normal,0,3,52,alto,Score 52/100 — risco alto. Regras acionadas: dispositivo utilizado por múltiplas contas; beneficiário recebe valores de muitas contas distintas; IP utilizado por múltiplas cont...
5,TX_00062859,687.0400,normal,0,2,32,medio,Score 32/100 — risco medio. Regras acionadas: dispositivo utilizado por múltiplas contas; IP utilizado por múltiplas contas distintas.
6,TX_00078572,310.9300,normal,0,2,32,medio,Score 32/100 — risco medio. Regras acionadas: dispositivo utilizado por múltiplas contas; IP utilizado por múltiplas contas distintas.
7,TX_00037550,299.0300,normal,0,2,32,medio,Score 32/100 — risco medio. Regras acionadas: dispositivo utilizado por múltiplas contas; IP utilizado por múltiplas contas distintas.
8,TX_00020042,"22,237.8000",new_account_high_value,1,7,100,critico,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...
9,TX_00001554,561.1600,normal,0,2,32,medio,Score 32/100 — risco medio. Regras acionadas: dispositivo utilizado por múltiplas contas; IP utilizado por múltiplas contas distintas.


## 9. Avaliação das Regras contra o Label Sintético

Como o dataset possui labels sintéticos, podemos avaliar o comportamento das regras.

A avaliação não representa performance real de produção. Ela serve para verificar se as regras capturam os cenários de fraude injetados de forma coerente.

In [13]:
rule_evaluation_rows = []

for rule_col in rule_flag_columns:
    temp = rules_scored.copy()
    triggered = temp[rule_col] == True

    qtd_acionadas = triggered.sum()
    qtd_fraudes_acionadas = temp.loc[triggered, "is_fraud"].sum()
    taxa_fraude_acionadas = temp.loc[triggered, "is_fraud"].mean() if qtd_acionadas > 0 else 0
    cobertura_fraude = (
        qtd_fraudes_acionadas / temp["is_fraud"].sum()
        if temp["is_fraud"].sum() > 0
        else 0
    )

    rule_evaluation_rows.append(
        {
            "regra": rule_col,
            "qtd_transacoes_acionadas": int(qtd_acionadas),
            "qtd_fraudes_sinteticas_acionadas": int(qtd_fraudes_acionadas),
            "taxa_fraude_entre_acionadas": taxa_fraude_acionadas,
            "cobertura_fraude_sintetica": cobertura_fraude,
            "valor_medio_acionadas": temp.loc[triggered, "valor"].mean() if qtd_acionadas > 0 else 0,
        }
    )

rule_evaluation = (
    pd.DataFrame(rule_evaluation_rows)
    .sort_values(["taxa_fraude_entre_acionadas", "cobertura_fraude_sintetica"], ascending=False)
)

rule_evaluation

,regra,qtd_transacoes_acionadas,qtd_fraudes_sinteticas_acionadas,taxa_fraude_entre_acionadas,cobertura_fraude_sintetica,valor_medio_acionadas
7,R008_muitos_beneficiarios_no_dia,1200,1200,1.0000,0.1714,"1,875.9103"
6,R007_rajada_transacional_horaria,1183,1183,1.0000,0.1690,"1,871.8059"
2,R003_conta_nova_alto_valor,1944,1506,0.7747,0.2151,"15,944.1989"
0,R001_alto_valor_transacional,4000,3012,0.7530,0.4303,"11,708.3042"
1,R002_valor_acima_limite_diario,4848,2679,0.5526,0.3827,"8,822.9442"
9,R010_canal_digital_alto_valor,7120,3651,0.5128,0.5216,"6,825.9760"
5,R006_rede_ou_device_alto_risco,9517,2990,0.3142,0.4271,"1,685.0548"
4,R005_beneficiario_concentrador,31469,4083,0.1297,0.5833,"1,179.1071"
3,R004_dispositivo_compartilhado,79996,7000,0.0875,1.0000,"1,012.5776"
8,R009_ip_compartilhado_multiplas_contas,80000,7000,0.0875,1.0000,"1,012.5719"


In [14]:
risk_band_evaluation = (
    rules_scored
    .groupby("risk_band")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        qtd_alertas=("alerta_gerado", "sum"),
        taxa_fraude_sintetica=("is_fraud", "mean"),
        valor_medio=("valor", "mean"),
        qtd_cenarios=("fraud_scenario", "nunique"),
        media_regras_acionadas=("qtd_regras_acionadas", "mean"),
    )
    .reset_index()
)

risk_order = {"baixo": 1, "medio": 2, "alto": 3, "critico": 4}
risk_band_evaluation["ordem"] = risk_band_evaluation["risk_band"].map(risk_order)
risk_band_evaluation = risk_band_evaluation.sort_values("ordem").drop(columns="ordem")

risk_band_evaluation

,risk_band,qtd_transacoes,qtd_alertas,taxa_fraude_sintetica,valor_medio,qtd_cenarios,media_regras_acionadas
1,baixo,3,0,0.0000,496.4767,1,1.0000
3,medio,40494,40494,0.0012,397.9864,3,2.0384
0,alto,34876,34876,0.0941,581.6673,7,3.1850
2,critico,4627,4627,0.7930,"9,639.5003",7,5.9764


In [15]:
scenario_rule_summary = (
    rules_scored
    .groupby("fraud_scenario")
    .agg(
        qtd_transacoes=("transacao_id", "count"),
        taxa_alerta=("alerta_gerado", "mean"),
        score_medio=("rule_score", "mean"),
        score_p95=("rule_score", lambda x: x.quantile(0.95)),
        media_regras_acionadas=("qtd_regras_acionadas", "mean"),
        taxa_fraude_sintetica=("is_fraud", "mean"),
    )
    .reset_index()
    .sort_values("score_medio", ascending=False)
)

scenario_rule_summary

,fraud_scenario,qtd_transacoes,taxa_alerta,score_medio,score_p95,media_regras_acionadas,taxa_fraude_sintetica
3,coordinated_network,1000,1.0000,94.7900,100.0000,6.6540,1.0000
4,new_account_high_value,1000,1.0000,91.7790,100.0000,6.2280,1.0000
2,burst_transactions,1200,1.0000,83.0525,100.0000,5.4242,1.0000
1,bridge_account,800,1.0000,72.9125,100.0000,5.0587,1.0000
0,beneficiary_concentrator,1400,1.0000,61.3064,87.0000,3.8107,1.0000
6,shared_device_ring,1600,1.0000,54.4744,66.0000,3.4569,1.0000
5,normal,73000,1.0000,41.8222,64.0000,2.5613,0.0000


## 10. Priorização de Alertas

Nesta etapa será criada uma visão de alertas priorizados.

A priorização considera:

- score da regra;
- quantidade de regras acionadas;
- valor da transação;
- risco técnico;
- cenário sintético;
- explicação textual.

Essa tabela será uma das principais saídas do notebook.

In [16]:
alert_columns = [
    "transacao_id",
    "conta_origem_id",
    "cliente_id",
    "beneficiario_id",
    "device_id",
    "ip_id",
    "valor",
    "data_hora",
    "tipo_transacao",
    "canal",
    "fraud_scenario",
    "is_fraud",
    "idade_conta_dias",
    "device_contas_distintas",
    "benef_contas_origem_distintas",
    "ip_contas_distintas",
    "qtd_transacoes_conta_hora",
    "qtd_beneficiarios_conta_dia",
    "qtd_regras_acionadas",
    "rule_score",
    "risk_band",
    "alerta_gerado",
    "alert_explanation",
    *rule_flag_columns,
]

alerts = (
    rules_scored
    .loc[rules_scored["alerta_gerado"], alert_columns]
    .sort_values(
        ["rule_score", "qtd_regras_acionadas", "valor"],
        ascending=False,
    )
    .reset_index(drop=True)
)

alerts.head(30)

,transacao_id,conta_origem_id,cliente_id,beneficiario_id,device_id,ip_id,valor,data_hora,tipo_transacao,canal,fraud_scenario,is_fraud,idade_conta_dias,device_contas_distintas,benef_contas_origem_distintas,ip_contas_distintas,qtd_transacoes_conta_hora,qtd_beneficiarios_conta_dia,qtd_regras_acionadas,rule_score,risk_band,alerta_gerado,alert_explanation,R001_alto_valor_transacional,R002_valor_acima_limite_diario,R003_conta_nova_alto_valor,R004_dispositivo_compartilhado,R005_beneficiario_concentrador,R006_rede_ou_device_alto_risco,R007_rajada_transacional_horaria,R008_muitos_beneficiarios_no_dia,R009_ip_compartilhado_multiplas_contas,R010_canal_digital_alto_valor
0,TX_00065230,CTA_003735,CLI_004112,BEN_000828,DEV_004123,IP_001741,"3,108.9600",2025-11-20 21:37:00,pix,app,burst_transactions,1,683,13,25,28,12,30,9,100,critico,True,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; dispositivo u...,True,True,False,True,True,True,True,True,True,True
1,TX_00040529,CTA_005726,CLI_003899,BEN_003230,DEV_002736,IP_002522,"43,810.5600",2025-04-03 20:10:24,ted,internet_banking,new_account_high_value,1,0,76,29,22,1,1,8,100,critico,True,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...,True,True,True,True,True,True,False,False,True,True
2,TX_00023417,CTA_000290,CLI_003463,BEN_003007,DEV_000999,IP_002541,"42,646.3200",2025-09-21 01:21:27,pix,internet_banking,new_account_high_value,1,0,25,30,31,1,1,8,100,critico,True,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...,True,True,True,True,True,True,False,False,True,True
3,TX_00077782,CTA_002140,CLI_003790,BEN_000486,DEV_000198,IP_000466,"41,629.4300",2025-05-28 11:20:46,pix,api,new_account_high_value,1,0,17,28,25,1,1,8,100,critico,True,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...,True,True,True,True,True,True,False,False,True,True
4,TX_00034217,CTA_000199,CLI_003295,BEN_002481,DEV_003007,IP_002686,"39,474.8100",2025-01-01 20:19:14,ted,app,new_account_high_value,1,0,15,26,36,1,1,8,100,critico,True,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...,True,True,True,True,True,True,False,False,True,True
5,TX_00005821,CTA_003299,CLI_004181,BEN_000430,DEV_000552,IP_002978,"38,159.6000",2025-08-05 10:28:07,pix,api,new_account_high_value,1,0,21,30,27,1,1,8,100,critico,True,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...,True,True,True,True,True,True,False,False,True,True
6,TX_00053067,CTA_004187,CLI_000653,BEN_000667,DEV_002517,IP_002154,"35,328.1100",2025-11-01 20:57:41,pix,internet_banking,new_account_high_value,1,0,13,28,30,1,1,8,100,critico,True,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...,True,True,True,True,True,True,False,False,True,True
7,TX_00010621,CTA_004530,CLI_004765,BEN_003447,DEV_004116,IP_001161,"34,174.8400",2025-07-03 06:08:40,ted,app,new_account_high_value,1,0,19,25,22,1,1,8,100,critico,True,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...,True,True,True,True,True,True,False,False,True,True
8,TX_00069030,CTA_002992,CLI_001417,BEN_002264,DEV_003166,IP_001708,"34,113.2500",2025-

In [17]:
critical_alerts = alerts.loc[alerts["risk_band"] == "critico"].copy()

critical_alerts[
    [
        "transacao_id",
        "conta_origem_id",
        "valor",
        "fraud_scenario",
        "qtd_regras_acionadas",
        "rule_score",
        "risk_band",
        "alert_explanation",
    ]
].head(20)

,transacao_id,conta_origem_id,valor,fraud_scenario,qtd_regras_acionadas,rule_score,risk_band,alert_explanation
0,TX_00065230,CTA_003735,"3,108.9600",burst_transactions,9,100,critico,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; dispositivo u...
1,TX_00040529,CTA_005726,"43,810.5600",new_account_high_value,8,100,critico,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...
2,TX_00023417,CTA_000290,"42,646.3200",new_account_high_value,8,100,critico,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...
3,TX_00077782,CTA_002140,"41,629.4300",new_account_high_value,8,100,critico,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...
4,TX_00034217,CTA_000199,"39,474.8100",new_account_high_value,8,100,critico,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...
5,TX_00005821,CTA_003299,"38,159.6000",new_account_high_value,8,100,critico,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...
6,TX_00053067,CTA_004187,"35,328.1100",new_account_high_value,8,100,critico,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...
7,TX_00010621,CTA_004530,"34,174.8400",new_account_high_value,8,100,critico,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...
8,TX_00069030,CTA_002992,"34,113.2500",new_account_high_value,8,100,critico,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...
9,TX_00006876,CTA_001148,"33,619.6300",new_account_high_value,8,100,critico,Score 100/100 — risco critico. Regras acionadas: valor da transação acima do percentil 95 da base; valor representa parcela elevada do limite transacional diário; conta nova re...


## 11. Visão Agregada por Conta, Dispositivo e Beneficiário

Além da priorização por transação, é útil agregar os alertas por entidades.

Essa visão prepara o caminho para o próximo notebook de modelagem em grafo, onde contas, dispositivos e beneficiários serão tratados como nós conectados por transações.

In [18]:
account_alert_summary = (
    alerts
    .groupby("conta_origem_id")
    .agg(
        qtd_alertas=("transacao_id", "count"),
        score_medio=("rule_score", "mean"),
        score_max=("rule_score", "max"),
        valor_total_alertado=("valor", "sum"),
        qtd_beneficiarios=("beneficiario_id", "nunique"),
        qtd_devices=("device_id", "nunique"),
        qtd_ips=("ip_id", "nunique"),
        taxa_fraude_sintetica=("is_fraud", "mean"),
    )
    .reset_index()
    .sort_values(["score_max", "qtd_alertas", "valor_total_alertado"], ascending=False)
)

device_alert_summary = (
    alerts
    .groupby("device_id")
    .agg(
        qtd_alertas=("transacao_id", "count"),
        contas_distintas=("conta_origem_id", "nunique"),
        score_medio=("rule_score", "mean"),
        score_max=("rule_score", "max"),
        valor_total_alertado=("valor", "sum"),
        taxa_fraude_sintetica=("is_fraud", "mean"),
    )
    .reset_index()
    .sort_values(["contas_distintas", "score_max", "qtd_alertas"], ascending=False)
)

beneficiary_alert_summary = (
    alerts
    .groupby("beneficiario_id")
    .agg(
        qtd_alertas=("transacao_id", "count"),
        contas_origem_distintas=("conta_origem_id", "nunique"),
        score_medio=("rule_score", "mean"),
        score_max=("rule_score", "max"),
        valor_total_alertado=("valor", "sum"),
        taxa_fraude_sintetica=("is_fraud", "mean"),
    )
    .reset_index()
    .sort_values(["contas_origem_distintas", "score_max", "qtd_alertas"], ascending=False)
)

display(account_alert_summary.head(15))
display(device_alert_summary.head(15))
display(beneficiary_alert_summary.head(15))

,conta_origem_id,qtd_alertas,score_medio,score_max,valor_total_alertado,qtd_beneficiarios,qtd_devices,qtd_ips,taxa_fraude_sintetica
2508,CTA_002509,86,82.2791,100,"525,957.0600",85,85,84,0.8953
663,CTA_000664,85,69.0353,100,"590,316.0600",84,84,84,0.9294
5334,CTA_005335,81,63.8765,100,"457,827.2000",81,80,79,0.8025
1133,CTA_001134,79,80.4557,100,"433,077.0100",78,79,79,0.8354
5359,CTA_005360,79,70.6203,100,"409,791.8000",78,77,78,0.8354
2394,CTA_002395,78,64.4615,100,"471,438.7500",77,77,77,0.8462
1975,CTA_001976,76,70.3421,100,"411,664.3900",75,76,76,0.8684
874,CTA_000875,75,67.9600,100,"385,964.2600",75,74,73,0.8400
4447,CTA_004448,66,67.8333,100,"379,933.8100",66,66,66,0.8788
3280,CTA_003281,55,74.2909,100,"92,931.9300",54,55,55,0.6909


,device_id,qtd_alertas,contas_distintas,score_medio,score_max,valor_total_alertado,taxa_fraude_sintetica
855,DEV_000856,150,147,54.4733,96,"74,310.8300",0.8933
4006,DEV_004007,145,143,53.6897,100,"60,139.8700",0.9241
3931,DEV_003932,100,99,57.0400,100,"46,842.8900",0.8100
2417,DEV_002418,97,96,54.7835,78,"31,917.1200",0.8351
2047,DEV_002048,95,91,54.4737,93,"55,695.3100",0.7474
101,DEV_000102,89,88,55.0674,89,"40,333.0900",0.7978
2458,DEV_002459,88,88,55.1705,89,"41,954.5400",0.7727
1271,DEV_001272,87,86,53.7011,100,"82,371.4700",0.8276
3862,DEV_003863,86,86,54.8721,86,"34,586.0800",0.8372
360,DEV_000361,83,83,56.0723,100,"55,482.9900",0.7108


,beneficiario_id,qtd_alertas,contas_origem_distintas,score_medio,score_max,valor_total_alertado,taxa_fraude_sintetica
2998,BEN_002999,108,106,61.4537,100,"180,489.2200",0.7870
3122,BEN_003123,108,105,59.6296,100,"140,199.2600",0.7315
3030,BEN_003031,103,102,61.6699,100,"185,369.5200",0.8058
267,BEN_000268,103,102,60.5825,97,"136,141.7500",0.8447
1263,BEN_001264,97,97,59.1856,97,"127,914.9900",0.7216
1790,BEN_001791,95,95,59.6000,100,"126,121.5500",0.7474
351,BEN_000352,94,94,60.9149,97,"140,127.4400",0.8511
997,BEN_000998,93,93,57.3763,96,"95,672.9300",0.7849
2786,BEN_002787,94,93,61.1277,87,"152,518.1000",0.7553
2456,BEN_002457,93,93,57.7312,87,"105,149.3000",0.7204


## 12. Síntese Executiva dos Resultados

Nesta etapa será criada uma tabela de síntese com os principais resultados do motor de regras.

Essa visão ajuda a comunicar o valor analítico do notebook de forma executiva.

In [19]:
executive_summary = pd.DataFrame(
    [
        {
            "indicador": "Total de transações analisadas",
            "valor": len(rules_scored),
            "interpretacao": "Base transacional sintética usada pelo motor de regras.",
        },
        {
            "indicador": "Total de alertas gerados",
            "valor": int(rules_scored["alerta_gerado"].sum()),
            "interpretacao": "Transações com score mínimo para investigação.",
        },
        {
            "indicador": "Taxa de alertas",
            "valor": rules_scored["alerta_gerado"].mean(),
            "interpretacao": "Proporção de transações priorizadas pelo motor de regras.",
        },
        {
            "indicador": "Score médio dos alertas",
            "valor": alerts["rule_score"].mean(),
            "interpretacao": "Pontuação média entre transações alertadas.",
        },
        {
            "indicador": "Alertas críticos",
            "valor": int((alerts["risk_band"] == "critico").sum()),
            "interpretacao": "Transações com score igual ou superior a 70.",
        },
        {
            "indicador": "Taxa sintética de fraude entre alertas",
            "valor": alerts["is_fraud"].mean(),
            "interpretacao": "Validação exploratória contra o label sintético.",
        },
        {
            "indicador": "Contas com alertas",
            "valor": alerts["conta_origem_id"].nunique(),
            "interpretacao": "Quantidade de contas distintas sinalizadas.",
        },
        {
            "indicador": "Dispositivos com alertas",
            "valor": alerts["device_id"].nunique(),
            "interpretacao": "Quantidade de dispositivos envolvidos em alertas.",
        },
        {
            "indicador": "Beneficiários com alertas",
            "valor": alerts["beneficiario_id"].nunique(),
            "interpretacao": "Quantidade de beneficiários envolvidos em alertas.",
        },
    ]
)

executive_summary

,indicador,valor,interpretacao
0,Total de transações analisadas,"80,000.0000",Base transacional sintética usada pelo motor de regras.
1,Total de alertas gerados,"79,997.0000",Transações com score mínimo para investigação.
2,Taxa de alertas,1.0000,Proporção de transações priorizadas pelo motor de regras.
3,Score médio dos alertas,44.6333,Pontuação média entre transações alertadas.
4,Alertas críticos,"4,627.0000",Transações com score igual ou superior a 70.
5,Taxa sintética de fraude entre alertas,0.0875,Validação exploratória contra o label sintético.
6,Contas com alertas,"6,000.0000",Quantidade de contas distintas sinalizadas.
7,Dispositivos com alertas,"4,500.0000",Quantidade de dispositivos envolvidos em alertas.
8,Beneficiários com alertas,"3,500.0000",Quantidade de beneficiários envolvidos em alertas.


## 13. Exportação dos Artefatos do Motor de Regras

Nesta etapa serão exportados os principais artefatos do notebook:

- catálogo de regras;
- base de transações com score;
- alertas priorizados;
- resumo por conta;
- resumo por dispositivo;
- resumo por beneficiário;
- relatório executivo em Markdown.

Os arquivos em `data/03-gold/` são dados derivados e podem ser recriados pelo notebook.  
Os relatórios em `docs/` e `artifacts/reports/` ajudam na documentação do projeto.

In [20]:
rules_catalog_path = DOCS_DIR / "antifraud_rules_catalog.md"
rules_engine_report_path = DOCS_DIR / "rules_engine_antifraud_summary.md"

rules_scored_path = GOLD_DIR / "transactions_with_rule_scores.parquet"
alerts_path = GOLD_DIR / "antifraud_alerts.parquet"
account_alert_summary_path = GOLD_DIR / "account_alert_summary.parquet"
device_alert_summary_path = GOLD_DIR / "device_alert_summary.parquet"
beneficiary_alert_summary_path = GOLD_DIR / "beneficiary_alert_summary.parquet"

alerts_sample_csv_path = REPORTS_DIR / "antifraud_alerts_sample.csv"
critical_alerts_csv_path = REPORTS_DIR / "critical_antifraud_alerts.csv"
rule_evaluation_csv_path = REPORTS_DIR / "rule_evaluation_summary.csv"

rules_scored.to_parquet(rules_scored_path, index=False)
alerts.to_parquet(alerts_path, index=False)
account_alert_summary.to_parquet(account_alert_summary_path, index=False)
device_alert_summary.to_parquet(device_alert_summary_path, index=False)
beneficiary_alert_summary.to_parquet(beneficiary_alert_summary_path, index=False)

alerts.head(100).to_csv(alerts_sample_csv_path, index=False, encoding="utf-8")
critical_alerts.head(100).to_csv(critical_alerts_csv_path, index=False, encoding="utf-8")
rule_evaluation.to_csv(rule_evaluation_csv_path, index=False, encoding="utf-8")

rules_catalog_md = "# Catálogo de Regras Antifraude\n\n"
rules_catalog_md += rules_catalog.to_markdown(index=False)
rules_catalog_md += "\n"

rules_catalog_path.write_text(rules_catalog_md, encoding="utf-8")

print("Artefatos exportados com sucesso:")
print(f"- {rules_scored_path}")
print(f"- {alerts_path}")
print(f"- {account_alert_summary_path}")
print(f"- {device_alert_summary_path}")
print(f"- {beneficiary_alert_summary_path}")
print(f"- {alerts_sample_csv_path}")
print(f"- {critical_alerts_csv_path}")
print(f"- {rule_evaluation_csv_path}")
print(f"- {rules_catalog_path}")

Artefatos exportados com sucesso:
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold\transactions_with_rule_scores.parquet
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold\antifraud_alerts.parquet
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold\account_alert_summary.parquet
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold\device_alert_summary.parquet
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\data\03-gold\beneficiary_alert_summary.parquet
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports\antifraud_alerts_sample.csv
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports\critical_antifraud_alerts.csv
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\artifacts\reports\rule_evaluation_summary.csv
- d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs\antifraud_rules_catalog.md


In [21]:
def safe_markdown_table(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except ImportError:
        return df.to_csv(index=False)


report_lines = [
    "# Rules Engine Antifraud — Resumo Executivo",
    "",
    "Este documento consolida os principais resultados do Notebook 03.",
    "",
    "## Objetivo",
    "",
    "Construir um motor de regras antifraude explicável para priorização de transações suspeitas em uma base sintética.",
    "",
    "## Síntese Executiva",
    "",
    safe_markdown_table(executive_summary),
    "",
    "## Avaliação por Regra",
    "",
    safe_markdown_table(rule_evaluation),
    "",
    "## Avaliação por Faixa de Risco",
    "",
    safe_markdown_table(risk_band_evaluation),
    "",
    "## Cenários Sintéticos e Score Médio",
    "",
    safe_markdown_table(scenario_rule_summary),
    "",
    "## Top Contas Alertadas",
    "",
    safe_markdown_table(account_alert_summary.head(10)),
    "",
    "## Top Dispositivos Alertados",
    "",
    safe_markdown_table(device_alert_summary.head(10)),
    "",
    "## Top Beneficiários Alertados",
    "",
    safe_markdown_table(beneficiary_alert_summary.head(10)),
    "",
    "## Observação",
    "",
    "Os resultados são derivados de dados sintéticos criados exclusivamente para fins educacionais, analíticos e de portfólio.",
    "O motor de regras não representa um sistema antifraude produtivo, mas demonstra uma abordagem explicável para investigação transacional.",
    "",
]

rules_engine_report_path.write_text("\n".join(report_lines), encoding="utf-8")

print(f"Relatório executivo exportado em: {rules_engine_report_path}")

Relatório executivo exportado em: d:\_DS-Projects\Data-Science\fraud-graph-analytics\docs\rules_engine_antifraud_summary.md


## 14. Conclusão Executiva do Notebook 03

Este notebook construiu a primeira versão do **motor de regras antifraude explicável** do projeto Fraud Graph Analytics.

Foram desenvolvidas regras para capturar sinais como:

- transações de alto valor;
- valor elevado em relação ao limite diário;
- contas novas com alto valor transacional;
- dispositivos compartilhados por múltiplas contas;
- beneficiários concentradores;
- IPs e dispositivos de maior risco;
- rajadas transacionais;
- pulverização de beneficiários;
- canais digitais com alto valor.

O motor gerou um score de risco por transação, classificando os casos em faixas de risco e criando explicações textuais para apoiar a investigação.

As saídas deste notebook serão usadas no próximo passo do projeto:

**Notebook 04 — Graph Modeling Neo4j**

No próximo notebook, as entidades e alertas serão convertidos em estruturas de grafo, permitindo investigar relações entre contas, transações, dispositivos, IPs, beneficiários e regras acionadas.